In [2]:
# Project Two Andre Miranda
# ProjectTwoDashboard.ipynb

# ---------------------------------------------------------
# JUPYTER DASH SETUP
# ---------------------------------------------------------
from jupyter_dash import JupyterDash
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl
import plotly.express as px
import base64
import pandas as pd
import re
from collections import Counter

from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

# ---------------------------------------------------------
# DATABASE CONNECTION
# ---------------------------------------------------------
username = "aacuser"
password = "CS340AndreMiranda"
shelter = AnimalShelter(username, password)

# Load full dataset
df = pd.DataFrame.from_records(shelter.read_all())

if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

# ---------------------------------------------------------
# LOGO ENCODING
# ---------------------------------------------------------
logo_path = "Grazioso Salvare Logo.png"
encoded_logo = base64.b64encode(open(logo_path, "rb").read()).decode()


# ---------------------------------------------------------
# DASH APP + CUSTOM HTML
# ---------------------------------------------------------
app = JupyterDash("GraziosoDashboard")

app.index_string = '''
<!DOCTYPE html>
<html>
    <head>
        {%metas%}
        <title>Grazioso Salvare Dashboard</title>
        {%favicon%}
        {%css%}

        <style>
            body {
                background-color: #f4f5f7;
                font-family: "Segoe UI", -apple-system, BlinkMacSystemFont, "Helvetica Neue", Arial, sans-serif;
                margin: 0;
                padding: 0;
                color: #1f2933;
            }

            .dashboard-container {
                max-width: 1200px;
                margin: 40px auto;
                padding: 30px;
                background-color: #ffffff;
                border-radius: 12px;
                box-shadow: 0 4px 12px rgba(15, 23, 42, 0.08);
            }

            .dashboard-title {
                text-align: center;
                font-size: 34px;
                font-weight: 700;
                color: #1f2933;
                margin-bottom: 4px;
            }

            .dashboard-subtitle {
                text-align: center;
                font-size: 14px;
                color: #6b7280;
                margin-bottom: 20px;
            }

            .filter-box {
                display: flex;
                justify-content: center;
                gap: 20px;
                margin-bottom: 20px;
            }

            .dash-table-container {
                margin-top: 20px;
                border-radius: 10px;
                overflow: hidden;
                border: 1px solid #e5e7eb;
            }

            #map-card {
                margin-top: 30px;
                padding: 18px 20px 22px 20px;
                background-color: #f9fafb;
                border-radius: 12px;
                border: 1px solid #e5e7eb;
                box-shadow: 0 2px 8px rgba(15, 23, 42, 0.04);
                position: relative;
            }

            #info-card {
                position: absolute;
                top: 20px;
                right: 20px;
                width: 220px;
                background-color: rgba(255,255,255,0.70);
                padding: 12px 14px;
                border-radius: 10px;
                box-shadow: 0 4px 10px rgba(0,0,0,0.15);
                backdrop-filter: blur(6px);
                font-size: 13px;
                line-height: 1.4;
                z-index: 9999;
            }
        </style>
    </head>

    <body>
        <div class="dashboard-container">
            {%app_entry%}
        </div>

        <footer>
            {%config%}
            {%scripts%}
            {%renderer%}
        </footer>
    </body>
</html>
'''

# ---------------------------------------------------------
# DASH LAYOUT
# ---------------------------------------------------------
app.layout = html.Div([

    # Logo + Title
    html.Center([
        html.Img(src=f"data:image/png;base64,{encoded_logo}", style={"height": "90px"}),
        html.H1("Grazioso Salvare Dashboard", className="dashboard-title"),
        html.Div("Created by Andre Miranda", className="dashboard-subtitle")
    ]),

    html.Hr(),

    # FILTER WIDGETS
    html.Div(
        className="filter-box",
        children=[
            dcc.RadioItems(
                # Rescue filter options based on the Rescue Type & Preferred Dog Breeds table
                id="rescue-filter",
                options=[
                    {"label": "Water Rescue", "value": "water"},
                    {"label": "Mountain/Wilderness", "value": "mountain"},
                    {"label": "Disaster/Tracking", "value": "disaster"},
                    {"label": "Reset", "value": "reset"}
                ],
                value="reset",
                inline=True,
                style={"fontSize": "16px"}
            )
        ]
    ),

    html.Hr(),

    # DATA TABLE
    html.Div(
        className="dash-table-container",
        children=[
            dash_table.DataTable(
                id="datatable-id",
                columns=[{"name": i, "id": i} for i in df.columns],
                data=df.to_dict("records"),
                page_size=10,
                sort_action="native",
                filter_action="native",
                row_selectable="single",
                selected_rows=[0],
                style_table={"height": "400px", "overflowY": "auto"},
                style_cell={"textAlign": "left", "padding": "6px 8px"},
            )
        ]
    ),

    html.Br(),
    html.Hr(),

    # MAP + PIE CHART
    html.Div(
        style={"display": "flex", "gap": "20px"},
        children=[
            html.Div(id="graph-id", style={"flex": "1"}),
            html.Div(
                id="map-card",
                style={"flex": "1"},
                children=[
                    html.Div(id="info-card"),
                    dcc.Loading(id="loading-map", children=html.Div(id="map-id"))
                ]
            )
        ]
    )
])

# ---------------------------------------------------------
# CALLBACK: FILTER DATA TABLE
# ---------------------------------------------------------
@app.callback(
    Output("datatable-id", "data"),
    Input("rescue-filter", "value")
)
# Age ranges based on Grazioso Salvare rescue training criteria
def update_table(filter_value):

    if filter_value == "water":
        records = shelter.filter_water_rescue()
    elif filter_value == "mountain":
        records = shelter.filter_mountain_rescue()
    elif filter_value == "disaster":
        records = shelter.filter_disaster_rescue()
    else:
        records = shelter.read_all()

    df_filtered = pd.DataFrame.from_records(records)
    if "_id" in df_filtered.columns:
        df_filtered.drop(columns=["_id"], inplace=True)

    return df_filtered.to_dict("records")

# ---------------------------------------------------------
# CALLBACK: PIE CHART
# ---------------------------------------------------------
def extract_breeds(breed_string):
    """
    Split mixed-breed strings into individual breeds.
    Handles '/', ',', 'Mix', and inconsistent spacing/casing.
    """
    if not isinstance(breed_string, str):
        return []

    # Split on '/', ',', or the word 'mix'
    parts = re.split(r"[/,]|mix", breed_string, flags=re.IGNORECASE)

    # Normalize each breed
    cleaned = [p.strip().title() for p in parts if p.strip()]

    return cleaned


@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graph(viewData):
    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if "breed" not in dff.columns or dff.empty:
        return []

    # -----------------------------------------
    # Extract and count individual breeds
    # -----------------------------------------
    all_breeds = []

    for breed_string in dff["breed"]:
        all_breeds.extend(extract_breeds(breed_string))

    if not all_breeds:
        return []

    breed_counts = Counter(all_breeds)

    # -----------------------------------------
    # Keep top 8 breeds, group the rest
    # -----------------------------------------
    top_n = 8
    top_breeds = breed_counts.most_common(top_n)

    final_labels = [b for b, _ in top_breeds]
    final_values = [c for _, c in top_breeds]

    # Group "Other Breeds"
    other_total = sum(count for breed, count in breed_counts.items()
                      if breed not in final_labels)

    if other_total > 0:
        final_labels.append("Other Breeds")
        final_values.append(other_total)

    # -----------------------------------------
    # Build pie chart
    # -----------------------------------------
    fig = px.pie(
        names=final_labels,
        values=final_values,
        title="Breed Distribution (Normalized)"
    )

    return [dcc.Graph(figure=fig)]



# ---------------------------------------------------------
# CALLBACK: MAP + INFO CARD
# ---------------------------------------------------------
@app.callback(
    [Output("map-id", "children"),
     Output("info-card", "children")],
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    if not viewData:
        return [], "No data available."

    dff = pd.DataFrame.from_dict(viewData)
    row = index[0] if index else 0

    # Extract coordinates
    lat = dff.iloc[row].get("location_lat")
    lon = dff.iloc[row].get("location_long")

    # Fallback if coordinates are missing or invalid
    if lat is None or lon is None or pd.isna(lat) or pd.isna(lon):
        fallback_info = html.Div([
            html.B("ID: "), str(dff.iloc[row]["animal_id"]), html.Br(),
            html.B("Name: "), str(dff.iloc[row]["name"]), html.Br(),
            html.Br(),
            html.I("Location data unavailable for this animal.")
        ])

        fallback_map = [
            dl.Map(
                style={"width": "100%", "height": "500px"},
                center=[30.2672, -97.7431],  # Austin, TX default center
                zoom=10,
                children=[dl.TileLayer()]
            )
        ]

        return fallback_map, fallback_info

    # Normal map + info card
    info = html.Div([
        html.B("ID: "), str(dff.iloc[row]["animal_id"]), html.Br(),
        html.B("Name: "), str(dff.iloc[row]["name"]), html.Br(),
        html.B("Type: "), str(dff.iloc[row]["animal_type"]), html.Br(),
        html.B("Breed: "), str(dff.iloc[row]["breed"]), html.Br(),
        html.B("Sex: "), str(dff.iloc[row]["sex_upon_outcome"]), html.Br(),
        html.B("Outcome: "), str(dff.iloc[row]["outcome_type"]), html.Br(),
        html.B("Age (weeks): "), str(dff.iloc[row]["age_upon_outcome_in_weeks"])
    ])

    map_component = [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[lat, lon],
            zoom=12,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(dff.iloc[row]["breed"]),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(dff.iloc[row]["name"])
                        ])
                    ]
                )
            ]
        )
    ]

    return map_component, info


# ---------------------------------------------------------
# RUN APP
# ---------------------------------------------------------
app.run_server(mode="external", port=8051)


[Connection Successful] Authenticated as 'aacuser' → Database: 'aac', Collection: 'animals'
Dash app running on https://citrusdesign-welcomehawaii-3000.codio.io/proxy/8051/
